# Tarea 03. Regresión y clasificación con Wine Quality

En esta tarea voy a usar los datos de vino tinto de Wine Quality para comparar regresión y clasificación. Primero intento predecir la calificación `quality` y después la convierto en dos clases: bueno si tiene 7 o más, y no bueno si tiene menos de 7.

En ambos casos uso regresión lineal y separo 80 % para entrenamiento y 20 % para prueba, con `random_state=42`. Me interesa ver qué cambia al modificar lo que quiero predecir.

## Dataset

Los datos vienen de [Wine Quality, de UCI](https://archive.ics.uci.edu/dataset/186/wine+quality). Son mediciones fisicoquímicas de vinos portugueses, acompañadas de una calificación de calidad en una escala de 0 a 10. Aquí trabajo solo con el archivo de vino tinto.

**Referencia:** Cortez, P., Cerdeira, A., Almeida, F., Matos, T. y Reis, J. (2009). [Wine Quality](https://doi.org/10.24432/C56S3T). UCI Machine Learning Repository. Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [ ]:
import pandas as pd                                  # manejo de datos
import numpy as np                                   # operaciones numéricas
import matplotlib.pyplot as plt                      # gráficas
from sklearn.model_selection import train_test_split # separación entrenamiento/prueba
from sklearn.linear_model import LinearRegression    # modelo solicitado
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")                       # el archivo usa ; como separador

df.head()

`quality` es la columna que voy a predecir. Las demás contienen las mediciones del vino y ya son numéricas. Antes de entrenar reviso cómo vienen los datos y cómo se distribuyen las calificaciones.

In [ ]:
print("dimensiones:", df.shape)                      # número de filas y columnas
print("\nvalores faltantes:")
print(df.isna().sum())
print("\ndistribución de quality:")
print(df["quality"].value_counts().sort_index())

plt.figure(figsize=(6, 4))
df["quality"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribución de la calidad")
plt.xlabel("calidad")
plt.ylabel("número de vinos")
plt.xticks(rotation=0)
plt.show()

El archivo de vino tinto tiene 1 599 registros y 12 columnas, sin faltantes. En la distribución me interesa revisar cuántos vinos hay en cada calificación, sobre todo en los extremos. Eso puede ayudar a entender después dónde se equivoca el modelo.

## 1. Regresión

Primero uso las 11 mediciones para predecir directamente `quality`.

También probaré algo más sencillo: asignar a todos los vinos el promedio de calidad del entrenamiento. Así tengo una referencia para ver si el modelo aporta algo al usar las características.

In [ ]:
x = df.drop(columns="quality")                       # variables de entrada
y = df["quality"]                                    # variable que queremos predecir

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)                                                    # separación 80/20 solicitada

print("entrenamiento:", len(x_train))
print("prueba:", len(x_test))

Quedan 1 279 vinos para entrenar y 320 para prueba. Uso una semilla fija para repetir la misma separación al volver a ejecutar el notebook.

In [ ]:
modelo_reg = LinearRegression()
modelo_reg.fit(x_train, y_train)                     # ajusta la recta usando entrenamiento

y_pred = modelo_reg.predict(x_test)                  # predice quality en los datos de prueba

rmse_reg = np.sqrt(mean_squared_error(y_test, y_pred))
r2_reg = r2_score(y_test, y_pred)

print("rmse regresión lineal:", round(rmse_reg, 4))
print("r² regresión lineal:", round(r2_reg, 4))

Reviso el RMSE para medir el error en puntos de calidad y el R² para comparar el ajuste. Un RMSE menor y un R² mayor indican un mejor resultado en esta prueba. Ahora compararé esos valores con la referencia del promedio.

In [ ]:
promedio_train = y_train.mean()                      # promedio calculado solo con entrenamiento
pred_promedio = np.full(len(y_test), promedio_train) # misma predicción para todos los vinos

rmse_promedio = np.sqrt(mean_squared_error(y_test, pred_promedio))
r2_promedio = r2_score(y_test, pred_promedio)

comparacion_reg = pd.DataFrame({
    "modelo": ["regresión lineal", "siempre promedio"],
    "rmse": [rmse_reg, rmse_promedio],
    "r²": [r2_reg, r2_promedio]
})

display(comparacion_reg.round(4))

Aquí comparo la regresión con predecir siempre el promedio. Si el modelo reduce el RMSE, usar las mediciones del vino habrá servido para mejorar esa predicción básica.

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.title("Calidad real contra calidad predicha")
plt.xlabel("calidad real")
plt.ylabel("calidad predicha")
plt.show()

La diagonal marca dónde coinciden la calidad real y la predicha. Cuanto más lejos quede un punto, mayor es el error. Me interesa revisar si el modelo se acerca a las calificaciones bajas y altas o si termina prediciendo casi siempre valores intermedios.

## 2. Clasificación con regresión lineal

Ahora cambio el objetivo: quiero decidir si un vino entra en la categoría de bueno. Defino `bueno = 1` cuando `quality >= 7` y `bueno = 0` en los demás casos.

Mantengo `LinearRegression`, pero esta vez la entreno con etiquetas 0 y 1. Como su salida sigue siendo continua, uso 0.5 como corte para asignar la clase.

In [ ]:
df["bueno"] = (df["quality"] >= 7).astype(int)       # convierte quality a una clase binaria

print(df["bueno"].value_counts())
print("\nporcentaje de vinos buenos:")
print(round(df["bueno"].mean() * 100, 2), "%")

plt.figure(figsize=(5, 4))
df["bueno"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribución de la variable bueno")
plt.xlabel("clase")
plt.ylabel("número de vinos")
plt.xticks([0, 1], ["no bueno", "bueno"], rotation=0)
plt.show()

Reviso cuántos vinos quedaron en cada clase. Si predominan los no buenos, responder siempre esa clase podría dar bastantes aciertos. Por eso también usaré esa estrategia como referencia.

In [ ]:
x_clas = df.drop(columns=["quality", "bueno"])       # mismas características originales
y_clas = df["bueno"]                                 # ahora el objetivo vale 0 o 1

x_train_c, x_test_c, y_train_c, y_test_c = train_test_split(
    x_clas,
    y_clas,
    test_size=0.20,
    random_state=42
)

modelo_clas = LinearRegression()
modelo_clas.fit(x_train_c, y_train_c)                # regresión aplicada a etiquetas 0 y 1

y_score = modelo_clas.predict(x_test_c)              # salida continua de la regresión
y_clase = (y_score >= 0.5).astype(int)               # convierte la salida en clase

accuracy_reg = accuracy_score(y_test_c, y_clase)

print("porcentaje de aciertos:", round(accuracy_reg * 100, 2), "%")

Las predicciones de 0.5 en adelante se convierten en clase 1 y las demás en 0. El porcentaje de aciertos resume cuántos vinos clasifiqué bien, pero quiero compararlo con responder siempre no bueno antes de valorar el resultado.

In [ ]:
pred_no_bueno = np.zeros(len(y_test_c), dtype=int)   # predice siempre la clase mayoritaria
accuracy_base = accuracy_score(y_test_c, pred_no_bueno)

fuera_rango = ((y_score < 0) | (y_score > 1)).sum()  # salidas imposibles como probabilidad

comparacion_clas = pd.DataFrame({
    "método": ["regresión + umbral 0.5", "siempre no bueno"],
    "aciertos (%)": [accuracy_reg * 100, accuracy_base * 100]
})

display(comparacion_clas.round(2))

print("predicciones fuera de [0, 1]:", fuera_rango)
print("mínimo predicho:", round(y_score.min(), 4))
print("máximo predicho:", round(y_score.max(), 4))

La tabla permite comparar los aciertos de las dos estrategias. También cuento cuántas salidas quedan fuera de 0 a 1: la regresión no limita sus predicciones a ese intervalo, así que no puedo leerlas directamente como probabilidades.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(y_score, bins=20, edgecolor="black")
plt.axvline(0.5, linestyle="--", label="umbral = 0.5")
plt.axvline(0, linestyle=":")
plt.axvline(1, linestyle=":")
plt.title("Salidas de la regresión para clasificación")
plt.xlabel("valor predicho")
plt.ylabel("frecuencia")
plt.legend()
plt.show()

La línea de 0.5 muestra el corte que estoy usando para clasificar. Las líneas de 0 y 1 ayudan a ubicar las salidas fuera de ese intervalo. Convertirlas en clases no cambia cómo funciona la regresión.

## Conclusión

Lo que cambia entre los dos casos es el objetivo. En regresión busco una calificación y reviso el error con RMSE y R². En clasificación busco una clase y necesito aplicar un corte a la salida del modelo.

Usar la misma regresión permite hacer esa comparación, pero sus salidas no son necesariamente probabilidades. Además, el porcentaje de aciertos necesita una referencia: si responder siempre no bueno da un resultado parecido, habría que revisar qué está aportando el modelo.